# N-BaIoT: Adaptive Fisher-Weighted Z Simulation with Rolling Statistics

This notebook reproduces the N-BaIoT experiment used to evaluate the adaptive client-specific anomaly-detection framework on benign traffic plus Mirai/BASHLITE attack traffic.

### Experiment flow
1. **Load N-BaIoT CSV files**, infer the physical device and attack labels from each filename, and construct disjoint benign and attack partitions.
2. **Split benign traffic chronologically within each device** into 30% general training, 10% baseline testing, 10% trusted enrollment, 10% PSO/client tuning, and 40% untouched final benign streaming evaluation.
3. **Create a bounded, balanced attack-tuning sample** while keeping the separate attack-test groups untouched for final evaluation.
4. **Build Fisher-weighted general device profiles** from benign training data, using the attack-tuning partition as the comparison distribution for feature weighting.
5. **Tune seven online adaptation parameters with PSO** using only enrollment/tuning data, controlled drift, and attack-tuning checkpoints.
6. **Enroll and calibrate client profiles**, then evaluate static versus adaptive behavior on the untouched benign and attack test data.
7. **Apply the causal 2-of-3 evidence rule** and save FPR, attack-detection, runtime, storage, and PSO summary tables.

### Reproducibility notes
- The notebook expects the N-BaIoT CSV directory and `profiles.py` at the configured relative paths.
- Benign partitions preserve chronological order within each physical device.
- Attack tuning and attack testing are separated before Fisher weighting/PSO so the final attack test is not used for optimization.
- Fisher feature weights remain fixed during online evaluation.
- Attack-test rows are **scored only** and never update the adaptive client profiles.
- PSO uses a fixed seed for repeatable optimization.


## 1. Load N-BaIoT and create disjoint temporal and attack splits

This section discovers the N-BaIoT traffic CSVs, extracts physical-device and attack labels from the filenames, validates/cleans the selected features, and limits very large files with evenly spaced sampling where configured.

Benign traffic is divided chronologically **inside each physical device** into:

- 30% general-profile training,
- 10% early baseline testing,
- 10% trusted client enrollment,
- 10% PSO/client tuning, and
- 40% untouched final benign stream.

Attack files are separated into a bounded **attack-tuning sample** and untouched **attack-test groups**. The attack-test portion is reserved for the final evaluation and is not used to learn Fisher weights or optimize PSO parameters.


In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

try:
    from IPython.display import display
except ImportError:
    display = print

from profiles import DeviceTypes, ClientProfiles

# Dataset path, deterministic controls, and the exact N-BaIoT feature order.
DATA_DIR = Path("archive")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)


ATTACK_TUNING_ROWS_PER_FILE = 500
ATTACK_TUNE_FRACTION = 0.20
RANDOM_STATE = 42


# Features used from the dataset columns
FEATURE_COLUMNS = [
    # Source MAC-IP traffic
    "MI_dir_L0.1_weight",
    "MI_dir_L0.1_mean",
    "MI_dir_L0.1_variance",
    # Source-host traffic
    "H_L0.1_weight",
    "H_L0.1_mean",
    "H_L0.1_variance",
    # Host-to-host channel
    "HH_L0.1_weight",
    "HH_L0.1_mean",
    "HH_L0.1_std",
    "HH_L0.1_magnitude",
    "HH_L0.1_radius",
    "HH_L0.1_covariance",
    "HH_L0.1_pcc",
    # Host-to-host jitter
    "HH_jit_L0.1_weight",
    "HH_jit_L0.1_mean",
    "HH_jit_L0.1_variance",
    # Host-port socket
    "HpHp_L0.1_weight",
    "HpHp_L0.1_mean",
    "HpHp_L0.1_std",
    "HpHp_L0.1_magnitude",
    "HpHp_L0.1_radius",
    "HpHp_L0.1_covariance",
    "HpHp_L0.1_pcc",
]

# Maping devices to their general device type
DEVICE_TYPE_MAP = {
    "device_1": "doorbell",  # Danmini Doorbell
    "device_2": "thermostat",  # Ecobee Thermostat
    "device_3": "doorbell",  # Ennio Doorbell
    "device_4": "baby_monitor",  # Philips B120N10
    "device_5": "security_camera",  # Provision PT-737E
    "device_6": "security_camera",  # Provision PT-838
    "device_7": "webcam",  # Samsung SNH-1011-N
    "device_8": "security_camera",  # SimpleHome XCS7-1002
    "device_9": "security_camera",  # SimpleHome XCS7-1003
}


def is_traffic_csv(path):
    """
    ========================================
    Ensures the CSV found is a traffic file
    ========================================
    """
    name = path.name.lower()
    return any(term in name for term in ("benign", "mirai", "gafgyt"))


def device_from_filename(path):
    """
    ========================================
    Extract the device from the path
    ========================================
    """
    prefix = path.name.split(".")[0]
    if not prefix.isdigit():
        raise ValueError(f"Cannot identify device from {path.name}")
    device_name = f"device_{prefix}"
    if device_name not in DEVICE_TYPE_MAP:
        raise ValueError(f"Unmapped N-BaIoT device: {device_name}")
    return device_name


def attack_family_from_filename(path):
    """
    ========================================
    Extract the attack family from path
    ========================================
    """
    name = path.name.lower()
    if "mirai" in name:
        return "Mirai"
    if "gafgyt" in name:
        return "Gafgyt"
    return "Unknown"


def attack_type_from_filename(path):
    """
    ========================================
    Extract the attack type from the path
    ========================================
    """
    name = path.name.lower()
    for attack_type in (
        "udpplain",
        "combo",
        "junk",
        "scan",
        "tcp",
        "ack",
        "syn",
        "udp",
    ):
        if attack_type in name:
            return attack_type
    return path.stem


def evenly_spaced(values, maximum_rows):
    """Select approximately evenly spaced row indices for a bounded sample."""
    values = np.asarray(values)
    if len(values) <= maximum_rows:
        return values
    positions = np.linspace(
        0,
        len(values) - 1,
        num=maximum_rows,
        dtype=int,
    )
    return values[positions]


# Ensure the directory is found
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Could not find the N-BaIoT folder: {DATA_DIR.resolve()}")

# Get all CSV files
# Discover only traffic CSVs; helper functions above decode labels from filenames.
csv_files = sorted(path for path in DATA_DIR.glob("*.csv") if is_traffic_csv(path))

# Ensure CSV files exist within directory
if not csv_files:
    raise FileNotFoundError("No N-BaIoT benign or attack CSV files were found.")



# Build benign rows plus separate attack-tuning and untouched attack-test partitions.
benign_parts = []
attack_tuning_parts = []
attack_test_groups = []

for file_number, path in enumerate(csv_files):
    # printing for debugging
    print(f"[{file_number + 1:>2}/{len(csv_files)}] {path.name}")

    # Try to get all rows from a CSV file
    try:
        rows = pd.read_csv(path, usecols=FEATURE_COLUMNS)
    except (
        OSError,
        UnicodeDecodeError,
        pd.errors.ParserError,
        pd.errors.EmptyDataError,
        ValueError,
    ) as error:
        print(f"Skipped {path.name}: {error}")
        continue

    # Apply used feature colmuns to the extracted rows
    rows[FEATURE_COLUMNS] = rows[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce")

    
    rows = rows.dropna(subset=FEATURE_COLUMNS).reset_index(drop=True)

    if rows.empty:
        print(f"Skipped {path.name}: no complete feature rows")
        continue

    device_name = device_from_filename(path)
    device_type = DEVICE_TYPE_MAP[device_name]

    if "benign" in path.name.lower():
        rows.insert(0, "device_name", device_name)
        rows.insert(1, "device_type", device_type)
        rows["position"] = np.arange(len(rows), dtype=int)
        benign_parts.append(rows)
        continue

    # Split row indices instead of copying one enormous attack dataframe.
    # This is the same reproducible 20%/80% train_test_split, performed once
    # per source CSV so every attack type is represented in both partitions.
    row_indices = np.arange(len(rows), dtype=int)
    tuning_indices, test_indices = train_test_split(
        row_indices,
        train_size=ATTACK_TUNE_FRACTION,
        random_state=RANDOM_STATE + file_number,
        shuffle=True,
    )
    tuning_indices = np.sort(tuning_indices)
    test_indices = np.sort(test_indices)

    tuning_indices = evenly_spaced(
        tuning_indices,
        ATTACK_TUNING_ROWS_PER_FILE,
    )

    attack_family = attack_family_from_filename(path)
    attack_type = attack_type_from_filename(path)

    tuning_rows = rows.iloc[tuning_indices].copy()
    tuning_rows.insert(0, "device_name", device_name)
    tuning_rows.insert(1, "device_type", device_type)
    tuning_rows.insert(2, "source_file", path.name)
    tuning_rows.insert(3, "attack_family", attack_family)
    tuning_rows.insert(4, "attack_type", attack_type)
    tuning_rows["attack_position"] = tuning_indices
    attack_tuning_parts.append(tuning_rows)

    # Float32 storage keeps the multi-million-row final attack partition
    # practical; scoring converts one source group to float64 at a time.
    attack_test_groups.append(
        {
            "device_name": device_name,
            "device_type": device_type,
            "source_file": path.name,
            "attack_family": attack_family,
            "attack_type": attack_type,
            "attack_positions": test_indices,
            "values": rows.iloc[test_indices][FEATURE_COLUMNS].to_numpy(
                dtype=np.float32,
            ),
        }
    )

if not benign_parts:
    raise ValueError("No benign N-BaIoT files were loaded.")
if not attack_tuning_parts or not attack_test_groups:
    raise ValueError("No attack N-BaIoT files were loaded.")

benign_df = pd.concat(benign_parts, ignore_index=True)
attack_tuning_df = pd.concat(attack_tuning_parts, ignore_index=True)
attack_test_windows = sum(len(group["values"]) for group in attack_test_groups)


def chronological_parts(dataframe):
    """Create disjoint temporal partitions inside every physical device."""
    names = [
        "general_train",
        "baseline_test",
        "enrollment",
        "tuning",
        "stream",
    ]
    fractions = [0.30, 0.10, 0.10, 0.10, 0.40]
    parts = {name: [] for name in names}
    summary = []

    for device_name, rows in dataframe.groupby("device_name", sort=False):
        rows = rows.sort_values("position", kind="stable").copy()
        cuts = np.floor(np.cumsum(fractions[:-1]) * len(rows)).astype(int)
        bounds = [0, *cuts.tolist(), len(rows)]
        counts = {}

        for name, start, stop in zip(names, bounds[:-1], bounds[1:]):
            part = rows.iloc[start:stop].copy()
            parts[name].append(part)
            counts[f"{name}_rows"] = len(part)

        summary.append(
            {
                "device_name": device_name,
                "device_type": rows["device_type"].iloc[0],
                "total_rows": len(rows),
                **counts,
            }
        )

    return (
        {name: pd.concat(items, ignore_index=True) for name, items in parts.items()},
        pd.DataFrame(summary),
    )


# Apply the fixed 30/10/10/10/40 chronological benign split per physical device.
benign_splits, device_split_summary = chronological_parts(benign_df)
general_train_df = benign_splits["general_train"]
baseline_test_df = benign_splits["baseline_test"]
enrollment_df = benign_splits["enrollment"]
tuning_df = benign_splits["tuning"]
stream_df = benign_splits["stream"]

split_summary = pd.DataFrame(
    [
        {"split": "general profile training", "rows": len(general_train_df)},
        {"split": "early baseline test", "rows": len(baseline_test_df)},
        {"split": "trusted client enrollment", "rows": len(enrollment_df)},
        {"split": "PSO and client tuning", "rows": len(tuning_df)},
        {
            "split": "untouched final benign stream",
            "rows": len(stream_df),
        },
        {
            "split": "balanced attack tuning sample",
            "rows": len(attack_tuning_df),
        },
        {
            "split": "untouched attack test",
            "rows": attack_test_windows,
        },
    ]
)

del benign_df, benign_parts, attack_tuning_parts, benign_splits

display(device_split_summary)
display(split_summary)


[ 1/89] 1.benign.csv
[ 2/89] 1.gafgyt.combo.csv
[ 3/89] 1.gafgyt.junk.csv
[ 4/89] 1.gafgyt.scan.csv
[ 5/89] 1.gafgyt.tcp.csv
[ 6/89] 1.gafgyt.udp.csv
[ 7/89] 1.mirai.ack.csv
[ 8/89] 1.mirai.scan.csv
[ 9/89] 1.mirai.syn.csv
[10/89] 1.mirai.udp.csv
[11/89] 1.mirai.udpplain.csv
[12/89] 2.benign.csv
[13/89] 2.gafgyt.combo.csv
[14/89] 2.gafgyt.junk.csv
[15/89] 2.gafgyt.scan.csv
[16/89] 2.gafgyt.tcp.csv
[17/89] 2.gafgyt.udp.csv
[18/89] 2.mirai.ack.csv
[19/89] 2.mirai.scan.csv
[20/89] 2.mirai.syn.csv
[21/89] 2.mirai.udp.csv
[22/89] 2.mirai.udpplain.csv
[23/89] 3.benign.csv
[24/89] 3.gafgyt.combo.csv
[25/89] 3.gafgyt.junk.csv
[26/89] 3.gafgyt.scan.csv
[27/89] 3.gafgyt.tcp.csv
[28/89] 3.gafgyt.udp.csv
[29/89] 4.benign.csv
[30/89] 4.gafgyt.combo.csv
[31/89] 4.gafgyt.junk.csv
[32/89] 4.gafgyt.scan.csv
[33/89] 4.gafgyt.tcp.csv
[34/89] 4.gafgyt.udp.csv
[35/89] 4.mirai.ack.csv
[36/89] 4.mirai.scan.csv
[37/89] 4.mirai.syn.csv
[38/89] 4.mirai.udp.csv
[39/89] 4.mirai.udpplain.csv
[40/89] 5.benign.csv
[

,device_name,device_type,total_rows,general_train_rows,baseline_test_rows,enrollment_rows,tuning_rows,stream_rows
0,device_1,doorbell,49548,14864,4955,4955,4954,19820
1,device_2,thermostat,13113,3933,1312,1311,1311,5246
2,device_3,doorbell,39100,11730,3910,3910,3910,15640
3,device_4,baby_monitor,175240,52572,17524,17524,17524,70096
4,device_5,security_camera,62154,18646,6215,6216,6215,24862
5,device_6,security_camera,98514,29554,9851,9852,9851,39406
6,device_7,webcam,52150,15645,5215,5215,5215,20860
7,device_8,security_camera,46585,13975,4659,4658,4659,18634
8,device_9,security_camera,19528,5858,1953,1953,1952,7812


,split,rows
0,general profile training,166777
1,early baseline test,55594
2,trusted client enrollment,55594
3,PSO and client tuning,55591
4,untouched final benign stream,222376
5,balanced attack tuning sample,40000
6,untouched attack test,5205371


## 2. Build Fisher-weighted general device-type profiles

For each N-BaIoT device type, this section learns the benign per-feature mean and standard deviation, fixed Fisher feature weights, and a 99th-percentile benign anomaly threshold.

The Fisher criterion compares benign general-training traffic with the disjoint attack-tuning sample. Features whose distributions separate benign and attack traffic more strongly receive larger normalized weights. The untouched attack-test groups are not used at this stage.

These Fisher weights remain fixed during the online experiment; only client-specific statistical profiles and thresholds are allowed to adapt.


In [2]:
# Fisher weights emphasize features that consistently separate benign and
# attack traffic. The weights are learned once from the disjoint attack-tuning
# split and remain fixed while each client's mean, standard deviation, and
# threshold adapt online.
SCORING_PERCENTILE = 99.0
SCORE_STD_FLOOR = 0.1
SCORING_FPR_TARGET = 0.01
SCORING_GROUP_DETECTION_TARGET = 0.95


def _safe_feature_stds(values, std_floor):
    """Return finite per-feature standard deviations with a minimum floor."""
    values = np.asarray(values, dtype=float)
    ddof = 1 if len(values) > 1 else 0
    stds = values.std(axis=0, ddof=ddof)
    stds = np.nan_to_num(
        stds,
        nan=std_floor,
        posinf=std_floor,
        neginf=std_floor,
    )
    return np.maximum(stds, std_floor)


def fisher_weights(benign_values, comparison_values):
    """Learn normalized Fisher separation weights without iterative training."""
    benign_values = np.asarray(benign_values, dtype=float)
    comparison_values = np.asarray(comparison_values, dtype=float)

    benign_means = benign_values.mean(axis=0)
    benign_stds = _safe_feature_stds(benign_values, SCORE_STD_FLOOR)
    comparison_means = comparison_values.mean(axis=0)
    comparison_stds = _safe_feature_stds(
        comparison_values,
        SCORE_STD_FLOOR,
    )

    pooled = np.sqrt(benign_stds**2 + comparison_stds**2)
    weights = np.abs(comparison_means - benign_means) / np.maximum(
        pooled,
        SCORE_STD_FLOOR,
    )
    weights = np.maximum(weights, np.finfo(float).eps)
    return weights / weights.mean()


def fisher_weighted_scores(values, means, stds, weights):
    """Return the Fisher-weighted mean absolute Z score."""
    values = np.asarray(values, dtype=float)
    one_row = values.ndim == 1
    values = np.atleast_2d(values)
    z_scores = np.abs(values - means) / np.maximum(
        stds,
        SCORE_STD_FLOOR,
    )
    scores = np.average(z_scores, axis=1, weights=weights)
    return float(scores[0]) if one_row else scores


def profile_arrays(profile):
    """Return profile means/stds as arrays aligned to FEATURE_COLUMNS."""
    means = np.array(
        [profile.feature_means[name] for name in FEATURE_COLUMNS],
        dtype=float,
    )
    stds = np.array(
        [profile.feature_stds[name] for name in FEATURE_COLUMNS],
        dtype=float,
    )
    return means, np.maximum(stds, SCORE_STD_FLOOR)


def profile_weights(profile):
    """Return profile Fisher weights as an array aligned to FEATURE_COLUMNS."""
    return np.array(
        [profile.feature_weights[name] for name in FEATURE_COLUMNS],
        dtype=float,
    )


def attack_detection_metrics(flags, rows):
    """Return micro, macro, and worst physical-device/attack-group detection."""
    scored = rows[["device_name", "attack_family", "attack_type"]].copy()
    scored["flagged"] = np.asarray(flags, dtype=bool)
    group_rates = scored.groupby(
        ["device_name", "attack_family", "attack_type"],
        sort=False,
    )["flagged"].mean()
    return (
        float(scored["flagged"].mean()),
        float(group_rates.mean()),
        float(group_rates.min()),
    )


# Train one general statistical profile and one fixed Fisher weight vector per device type.
general_profiles = {}
FISHER_THRESHOLD_BY_TYPE = {}
profile_rows = []
weight_rows = []
pooled_attack_values = attack_tuning_df[
    FEATURE_COLUMNS
].to_numpy(dtype=float)

# Fisher weights use only the disjoint attack-tuning sample; final attacks remain untouched.
for device_type, train_rows in general_train_df.groupby(
    "device_type",
    sort=False,
):
    train_values = train_rows[FEATURE_COLUMNS].to_numpy(dtype=float)
    validation_rows = tuning_df[tuning_df["device_type"] == device_type]
    validation_values = validation_rows[FEATURE_COLUMNS].to_numpy(dtype=float)
    attack_rows = attack_tuning_df[
        attack_tuning_df["device_type"] == device_type
    ]

    if len(attack_rows):
        comparison_values = attack_rows[
            FEATURE_COLUMNS
        ].to_numpy(dtype=float)
        weight_source = "device-type attack tuning"
    elif len(pooled_attack_values):
        comparison_values = pooled_attack_values
        weight_source = "pooled attack tuning fallback"
    else:
        raise ValueError(
            "Adaptive Fisher-weighted Z requires a non-empty attack-tuning split."
        )

    means = train_values.mean(axis=0)
    stds = _safe_feature_stds(train_values, SCORE_STD_FLOOR)
    weights = fisher_weights(train_values, comparison_values)

    train_scores = fisher_weighted_scores(
        train_values,
        means,
        stds,
        weights,
    )
    validation_scores = fisher_weighted_scores(
        validation_values,
        means,
        stds,
        weights,
    )
    attack_scores = fisher_weighted_scores(
        comparison_values,
        means,
        stds,
        weights,
    )
    threshold = float(
        np.percentile(train_scores, SCORING_PERCENTILE)
    )
    validation_fpr = float(
        np.mean(validation_scores > threshold)
    )

    # Report separation on the same attack-tuning rows used to learn the
    # weights. Final attack performance is still measured later on the untouched
    # attack-test partition.
    if len(attack_rows):
        micro, macro, worst_group = attack_detection_metrics(
            attack_scores > threshold,
            attack_rows,
        )
    else:
        micro = macro = worst_group = np.nan

    profile = DeviceTypes(
        device_type_name=device_type,
        feature_means=dict(zip(FEATURE_COLUMNS, means)),
        feature_stds=dict(zip(FEATURE_COLUMNS, stds)),
        feature_weights=dict(zip(FEATURE_COLUMNS, weights)),
        threshold=threshold,
        std_floor=SCORE_STD_FLOOR,
    )
    general_profiles[device_type] = profile
    FISHER_THRESHOLD_BY_TYPE[device_type] = threshold

    profile_rows.append({
        "device_type": device_type,
        "scoring_method": "Fisher-Weighted Z",
        "weight_source": weight_source,
        "training_rows": len(train_rows),
        "threshold": threshold,
        "validation_fpr": validation_fpr,
        "attack_micro_detection": micro,
        "attack_macro_detection": macro,
        "worst_group_detection": worst_group,
    })

    for feature, weight in zip(FEATURE_COLUMNS, weights):
        weight_rows.append({
            "device_type": device_type,
            "feature": feature,
            "fisher_weight": float(weight),
            "weight_source": weight_source,
        })

general_profile_summary = pd.DataFrame(profile_rows)
fisher_feature_weights = pd.DataFrame(weight_rows)
display(general_profile_summary)
display(
    fisher_feature_weights.sort_values(
        ["device_type", "fisher_weight"],
        ascending=[True, False],
        kind="stable",
    ).groupby("device_type", sort=False).head(5)
)


,device_type,scoring_method,weight_source,training_rows,threshold,validation_fpr,attack_micro_detection,attack_macro_detection,worst_group_detection
0,doorbell,Fisher-Weighted Z,device-type attack tuning,26594,2.013717,0.001128,1.00000,1.00000,1.000
1,thermostat,Fisher-Weighted Z,device-type attack tuning,3933,2.067388,0.001526,0.99860,0.99860,0.998
2,baby_monitor,Fisher-Weighted Z,device-type attack tuning,52572,4.409099,0.000000,0.99880,0.99880,0.998
3,security_camera,Fisher-Weighted Z,device-type attack tuning,68033,2.394713,0.000088,0.99945,0.99945,0.998
4,webcam,Fisher-Weighted Z,device-type attack tuning,15645,1.937117,0.005753,1.00000,1.00000,1.000


,device_type,feature,fisher_weight,weight_source
46,baby_monitor,MI_dir_L0.1_weight,3.725171,device-type attack tuning
49,baby_monitor,H_L0.1_weight,3.725171,device-type attack tuning
60,baby_monitor,HH_jit_L0.1_mean,2.662428,device-type attack tuning
52,baby_monitor,HH_L0.1_weight,2.171657,device-type attack tuning
59,baby_monitor,HH_jit_L0.1_weight,2.171657,device-type attack tuning
0,doorbell,MI_dir_L0.1_weight,3.126542,device-type attack tuning
3,doorbell,H_L0.1_weight,3.126541,device-type attack tuning
14,doorbell,HH_jit_L0.1_mean,2.687389,device-type attack tuning
6,doorbell,HH_L0.1_weight,2.105157,device-type attack tuning
13,doorbell,HH_jit_L0.1_weight,2.105157,device-type attack tuning


## 3. Define compact rolling-moment adaptive Fisher updates

Each physical client begins from the corresponding general device profile and adapts online using compact running statistics rather than retaining a history of accepted feature vectors.

Enrollment, stable updates, and drift confirmation share one Welford-style accumulator containing the accepted-window count, per-feature running mean, and per-feature $M_2$. During drift confirmation, only one scalar anomaly score per candidate window is additionally retained for threshold recalibration.

The decision paths are:

- **Enrollment:** trusted early benign windows initialize the client profile in fixed 64-window batches.
- **Stable:** sufficiently normal windows are accumulated and blended into the client profile at the PSO-selected batch size.
- **Drift candidate:** moderately elevated scores accumulate until the confirmation requirement is met; confirmed drift updates profile statistics and threshold.
- **Hard/blocked anomaly:** extreme behavior is excluded from adaptation to reduce self-poisoning.

Final anomaly decisions use a causal **2-of-3 evidence rule** over the latest raw flags.


In [3]:
# Enrollment stays fixed; PSO tunes the online adaptation controls per device type.
# Fixed safety/decision controls. PSO tunes the seven adaptation parameters defined later.
ENROLLMENT_BATCH_SIZE = 64
RELATIVE_STD_FLOOR = 0.60
DRIFT_WARNING_RATIO = 0.80
HARD_SCORE_RATIO = 3.0
THRESHOLD_ALPHA = 0.20
UPDATE_CLIP_SIGMA = 3.0

# Final decision smoothing requires at least two positive raw flags among the latest three.
EVIDENCE_WINDOW = 3
EVIDENCE_MIN_HITS = 2


def new_client_profile(client_id, device_type):
    """Initialize a client as a copy of its general device-type profile."""
    general = general_profiles[device_type]
    return ClientProfiles(
        client_id=client_id,
        device_type_name=device_type,
        feature_means=general.feature_means,
        feature_stds=general.feature_stds,
        feature_weights=general.feature_weights,
        threshold=general.threshold,
        std_floor=general.std_floor,
    )


def new_running_batch(score_capacity):
    """Create one compact accumulator shared by the mutually exclusive paths.

    The accumulator stores only a count, one running mean per feature, and one
    running M2 value per feature. Candidate drift scores use a bounded float64
    array so the threshold percentile can still be updated without storing full
    23-feature windows.
    """
    return {
        "mode": None,
        "count": 0,
        "mean": np.zeros(len(FEATURE_COLUMNS), dtype=np.float64),
        "m2": np.zeros(len(FEATURE_COLUMNS), dtype=np.float64),
        "score_capacity": int(score_capacity),
        "scores": None,
    }


def reset_running_batch(batch):
    """Clear the current enrollment/stable/drift accumulator in place."""
    batch["mode"] = None
    batch["count"] = 0
    batch["mean"].fill(0.0)
    batch["m2"].fill(0.0)
    batch["scores"] = None


def new_state(score_capacity):
    """Create the compact online state retained for one adaptive client."""
    return {
        # Stable/enrollment and drift accumulation are mutually exclusive, so
        # one running-moment state replaces the two raw-window buffers.
        "running_batch": new_running_batch(score_capacity),
        "batch_updates": 0,
        "confirmed_drifts": 0,
        "blocked_extremes": 0,
    }


def effective_arrays(profile, params):
    """Blend client and general statistics according to the anchor strength."""
    client_means, client_stds = profile_arrays(profile)
    general_means, general_stds = profile_arrays(
        general_profiles[profile.device_type_name]
    )
    anchor = float(params["anchor_strength"])
    means = (1.0 - anchor) * client_means + anchor * general_means
    variances = (
        (1.0 - anchor) * client_stds**2
        + anchor * general_stds**2
        + anchor * (1.0 - anchor) * (client_means - general_means) ** 2
    )
    return means, np.sqrt(np.maximum(variances, SCORE_STD_FLOOR**2))


def score_general(profile, values):
    """Score one feature vector using a fixed general device-type profile."""
    window = dict(zip(FEATURE_COLUMNS, np.asarray(values, dtype=float)))
    return float(profile.score_window(window))


def score_client(profile, values, params):
    """Score one feature vector using the anchored adaptive client profile."""
    general = general_profiles[profile.device_type_name]
    window = dict(zip(FEATURE_COLUMNS, np.asarray(values, dtype=float)))
    return float(
        profile.score_window(
            window,
            general_profile=general,
            anchor_strength=float(params["anchor_strength"]),
        )
    )


def score_client_matrix(profile, values, params):
    """Vectorized adaptive scoring used during tuning and threshold calibration."""
    means, stds = effective_arrays(profile, params)
    return fisher_weighted_scores(
        values,
        means,
        stds,
        profile_weights(profile),
    )


def add_running_observation(
    batch,
    mode,
    values,
    profile,
    params,
    score=None,
    clip_values=True,
):
    """Update exact per-feature batch moments without retaining the window."""
    if batch["mode"] != mode:
        reset_running_batch(batch)
        batch["mode"] = mode

    observation = np.asarray(values, dtype=np.float64)
    if observation.shape != (len(FEATURE_COLUMNS),):
        raise ValueError(
            f"Expected {len(FEATURE_COLUMNS)} features, got {observation.shape}."
        )

    # The profile is unchanged while a batch is accumulating, so clipping each
    # arriving observation is equivalent to clipping the full matrix immediately
    # before the old batch update.
    if clip_values:
        live_means, live_stds = effective_arrays(profile, params)
        observation = np.clip(
            observation,
            live_means - UPDATE_CLIP_SIGMA * live_stds,
            live_means + UPDATE_CLIP_SIGMA * live_stds,
        )

    new_count = batch["count"] + 1
    delta = observation - batch["mean"]
    batch["mean"] += delta / new_count
    delta_after = observation - batch["mean"]
    batch["m2"] += delta * delta_after
    batch["count"] = new_count

    if mode == "drift":
        if score is None:
            raise ValueError("A scalar score is required for drift accumulation.")
        if batch["scores"] is None:
            batch["scores"] = np.empty(
                batch["score_capacity"],
                dtype=np.float64,
            )
        if new_count > len(batch["scores"]):
            raise ValueError("Drift score accumulator capacity was exceeded.")
        batch["scores"][new_count - 1] = float(score)

    return new_count


def update_profile_from_moments(profile, batch, alpha):
    """Blend the current profile with the accumulated batch mean and variance."""
    if alpha <= 0.0 or batch["count"] == 0:
        return False

    batch_means = batch["mean"].copy()
    batch_variances = batch["m2"] / batch["count"]

    old_means, old_stds = profile_arrays(profile)
    new_means = (1.0 - alpha) * old_means + alpha * batch_means
    new_variances = (
        (1.0 - alpha) * old_stds**2
        + alpha * batch_variances
        + alpha * (1.0 - alpha) * (old_means - batch_means) ** 2
    )

    _, general_stds = profile_arrays(general_profiles[profile.device_type_name])
    minimum_stds = np.maximum(
        SCORE_STD_FLOOR,
        RELATIVE_STD_FLOOR * general_stds,
    )
    new_stds = np.maximum(np.sqrt(np.maximum(new_variances, 0.0)), minimum_stds)
    profile.feature_means = dict(zip(FEATURE_COLUMNS, new_means))
    profile.feature_stds = dict(zip(FEATURE_COLUMNS, new_stds))
    return True


def drift_evidence(profile, score):
    """Use only the Fisher score ratio to separate drift candidates from attacks."""
    safe = np.finfo(float).eps
    score_ratio = score / max(profile.threshold, safe)
    hard = score_ratio > HARD_SCORE_RATIO
    candidate = (
        not hard
        and score_ratio >= DRIFT_WARNING_RATIO
    )
    return candidate, hard, score_ratio


def recalibrate_threshold_from_scores(profile, scores, params):
    """Retain percentile-based calibration using compact scalar drift scores.

    Full drift windows are no longer retained, so the candidate percentile uses
    the scores observed while the confirmation batch was accumulating rather
    than rescoring every stored window after the profile update.
    """

    candidate = float(np.percentile(scores, SCORING_PERCENTILE))
    general_threshold = general_profiles[profile.device_type_name].threshold
    blended = (
        (1.0 - THRESHOLD_ALPHA) * profile.threshold
        + THRESHOLD_ALPHA * candidate
    )
    profile.threshold = float(
        np.clip(
            blended,
            general_threshold,
            float(params["threshold_ceiling"]) * general_threshold,
        )
    )


def process_observation(profile, values, params, state, trusted=False):
    """Score one window, then route it to a stable, drift, or blocked path."""
    score_start = time.perf_counter()
    score = score_client(profile, values, params)
    score_time_ms = (time.perf_counter() - score_start) * 1000
    threshold_before = float(profile.threshold)
    flagged = profile.is_anomalous(score)
    profile.window_count += 1

    candidate, hard, score_ratio = drift_evidence(profile, score)
    updated = False
    confirmed = False
    action = "no_update"
    update_start = time.perf_counter()
    batch = state["running_batch"]

    if trusted:
        # Enrollment is explicitly benign but remains clipped against the
        # anchored live profile to limit unusually large enrollment bursts.
        count = add_running_observation(
            batch,
            "enrollment",
            values,
            profile,
            params,
            clip_values=True,
        )
        action = "enrollment_accumulator"
        if count >= ENROLLMENT_BATCH_SIZE:
            updated = update_profile_from_moments(
                profile,
                batch,
                float(params["warmup_alpha"]),
            )
            reset_running_batch(batch)
            action = "enrollment_batch"

    elif hard:
        reset_running_batch(batch)
        state["blocked_extremes"] += 1
        action = "blocked_extreme"

    elif candidate:
        count = add_running_observation(
            batch,
            "drift",
            values,
            profile,
            params,
            score=score,
            clip_values=True,
        )
        action = "drift_accumulator"
        if count >= int(params["confirmation_windows"]):
            confirmed_scores = batch["scores"][:batch["count"]].copy()
            updated = update_profile_from_moments(
                profile,
                batch,
                float(params["drift_alpha"]),
            )
            if updated:
                recalibrate_threshold_from_scores(
                    profile,
                    confirmed_scores,
                    params,
                )
                state["confirmed_drifts"] += 1
                confirmed = True
            reset_running_batch(batch)
            action = "drift_confirmed"

    elif not flagged:
        count = add_running_observation(
            batch,
            "stable",
            values,
            profile,
            params,
            clip_values=True,
        )
        action = "stable_accumulator"
        if count >= int(params["stable_batch_size"]):
            updated = update_profile_from_moments(
                profile,
                batch,
                float(params["stable_alpha"]),
            )
            reset_running_batch(batch)
            action = "stable_batch"

    else:
        reset_running_batch(batch)
        action = "blocked_anomaly"

    if updated:
        state["batch_updates"] += 1

    return {
        "score": score,
        "threshold_before": threshold_before,
        "threshold_after": float(profile.threshold),
        "flagged": bool(flagged),
        "updated": bool(updated),
        "update_action": action,
        "drift_confirmed": bool(confirmed),
        "score_ratio": float(score_ratio),
        "score_time_ms": float(score_time_ms),
        "update_time_ms": float((time.perf_counter() - update_start) * 1000),
    }


def flush_enrollment(profile, params, state):
    """Apply any partial trusted-enrollment batch left after full batches."""
    batch = state["running_batch"]
    if batch["mode"] != "enrollment" or batch["count"] == 0:
        return False

    updated = update_profile_from_moments(
        profile,
        batch,
        float(params["warmup_alpha"]),
    )
    reset_running_batch(batch)
    state["batch_updates"] += int(updated)
    return updated


def controlled_drift(enrollment_values, tuning_values, general_stds):
    """Create a moderate broad shift used only while tuning adaptation."""
    direction = np.sign(
        np.median(tuning_values, axis=0) - np.median(enrollment_values, axis=0)
    )
    fallback = np.where(
        np.arange(tuning_values.shape[1]) % 2 == 0,
        1.0,
        -1.0,
    )
    direction = np.where(direction == 0.0, fallback, direction)
    return tuning_values + 0.85 * general_stds * direction


def add_evidence_flags(dataframe, group_columns, order_column):
    """Apply the causal final decision: at least two raw flags in the last three windows."""
    result = dataframe.copy()
    result["evidence_flagged"] = False
    for _, group in result.groupby(group_columns, sort=False):
        ordered = group.sort_values(order_column, kind="stable")
        hits = (
            ordered["flagged"].astype(int).rolling(EVIDENCE_WINDOW, min_periods=1).sum()
        )
        result.loc[ordered.index, "evidence_flagged"] = (
            hits >= EVIDENCE_MIN_HITS
        ).to_numpy()
    return result


## 4. Tune the seven adaptation parameters with PSO

PSO is run independently for each device type and tunes:

`warmup_alpha`, `stable_alpha`, `drift_alpha`, `anchor_strength`, `confirmation_windows`, `stable_batch_size`, and `threshold_ceiling`.

Each candidate is tested on held-out enrollment/tuning streams, a controlled moderate drift derived from tuning data, and attack-tuning checkpoints. The objective prioritizes low stable/drift FPR while penalizing candidates whose attack detection falls below the corresponding static reference performance.

Neither the final 40% benign stream nor the untouched attack-test groups are used during optimization.


In [4]:
# Cache NumPy arrays for repeated PSO evaluation and keep attack tuning separate from final testing.
ENROLLMENT_BY_CLIENT = {
    client: rows.sort_values("position", kind="stable")[FEATURE_COLUMNS].to_numpy(
        dtype=float
    )
    for client, rows in enrollment_df.groupby("device_name", sort=False)
}
TUNING_BY_CLIENT = {
    client: rows.sort_values("position", kind="stable")[FEATURE_COLUMNS].to_numpy(
        dtype=float
    )
    for client, rows in tuning_df.groupby("device_name", sort=False)
}
TYPE_BY_CLIENT = (
    enrollment_df[["device_name", "device_type"]]
    .drop_duplicates()
    .set_index("device_name")["device_type"]
    .to_dict()
)
CLIENTS_BY_TYPE = {
    device_type: sorted(
        client
        for client, client_type in TYPE_BY_CLIENT.items()
        if client_type == device_type
    )
    for device_type in general_profiles
}


ATTACK_TUNING_GROUPS_BY_CLIENT = {}
STATIC_GROUP_DETECTION = {}

for (
    client_id,
    device_type,
    attack_family,
    attack_type,
), rows in attack_tuning_df.groupby(
    [
        "device_name",
        "device_type",
        "attack_family",
        "attack_type",
    ],
    sort=False,
):
    values = rows[FEATURE_COLUMNS].to_numpy(dtype=float)
    key = (client_id, attack_family, attack_type)
    ATTACK_TUNING_GROUPS_BY_CLIENT.setdefault(
        client_id,
        [],
    ).append((key, values))

    general = general_profiles[device_type]
    means, stds = profile_arrays(general)
    scores = fisher_weighted_scores(
        values,
        means,
        stds,
        profile_weights(general),
    )
    STATIC_GROUP_DETECTION[key] = float(np.mean(scores > general.threshold))


def attack_checkpoint_rows(
    profile,
    client_id,
    device_type,
    params,
    checkpoint,
):
    """Measure every client/attack group at one adaptation checkpoint."""
    rows_out = []
    for group_key, values in ATTACK_TUNING_GROUPS_BY_CLIENT[client_id]:
        scores = score_client_matrix(profile, values, params)
        detection = float(np.mean(scores > profile.threshold))
        static_detection = STATIC_GROUP_DETECTION[group_key]
        minimum_detection = max(
            SCORING_GROUP_DETECTION_TARGET,
            min(0.99, static_detection - 0.01),
        )
        rows_out.append(
            {
                "device_type": device_type,
                "device_name": group_key[0],
                "attack_family": group_key[1],
                "attack_type": group_key[2],
                "checkpoint": checkpoint,
                "detection": detection,
                "static_detection": static_detection,
                "minimum_detection": minimum_detection,
                "shortfall": max(
                    0.0,
                    minimum_detection - detection,
                ),
            }
        )
    return rows_out


# Candidate scoring combines benign FPR, controlled drift, and attack-detection safeguards.
def simulate_candidate(device_type, params):
    """Evaluate FPR and worst attack groups without touching final test data."""
    stable_flags = []
    drift_flags = []
    stable_fpr_by_client = []
    drift_fpr_by_client = []
    attack_checkpoints = []

    for client_id in CLIENTS_BY_TYPE[device_type]:
        profile = new_client_profile(client_id, device_type)
        state = new_state(int(params["confirmation_windows"]))
        enrollment_values = ENROLLMENT_BY_CLIENT[client_id]
        tuning_values = TUNING_BY_CLIENT[client_id]

        for values in enrollment_values:
            process_observation(
                profile,
                values,
                params,
                state,
                trusted=True,
            )
        flush_enrollment(profile, params, state)
        attack_checkpoints.extend(
            attack_checkpoint_rows(
                profile,
                client_id,
                device_type,
                params,
                "after_enrollment",
            )
        )

        client_stable_flags = []
        for values in tuning_values:
            client_stable_flags.append(
                process_observation(
                    profile,
                    values,
                    params,
                    state,
                )["flagged"]
            )
        stable_flags.extend(client_stable_flags)
        stable_fpr_by_client.append(float(np.mean(client_stable_flags)))
        attack_checkpoints.extend(
            attack_checkpoint_rows(
                profile,
                client_id,
                device_type,
                params,
                "after_benign_tuning",
            )
        )

        _, general_stds = profile_arrays(general_profiles[device_type])
        shifted_values = controlled_drift(
            enrollment_values,
            tuning_values,
            general_stds,
        )
        client_drift_flags = []
        for values in shifted_values:
            client_drift_flags.append(
                process_observation(
                    profile,
                    values,
                    params,
                    state,
                )["flagged"]
            )
        drift_flags.extend(client_drift_flags)
        drift_fpr_by_client.append(float(np.mean(client_drift_flags)))
        attack_checkpoints.extend(
            attack_checkpoint_rows(
                profile,
                client_id,
                device_type,
                params,
                "after_controlled_drift",
            )
        )

    stable_fpr = float(np.mean(stable_flags))
    drift_fpr = float(np.mean(drift_flags))
    worst_client_stable_fpr = float(max(stable_fpr_by_client))
    worst_client_drift_fpr = float(max(drift_fpr_by_client))
    checkpoint_df = pd.DataFrame(attack_checkpoints)
    detection_rate = float(checkpoint_df["detection"].mean())
    worst_group_detection = float(checkpoint_df["detection"].min())
    mean_group_shortfall = float(checkpoint_df["shortfall"].mean())
    maximum_group_shortfall = float(checkpoint_df["shortfall"].max())

    # Detection penalties dominate FPR improvements whenever a candidate
    # over-specializes even one physical device or attack type.
    objective = (
        4.0 * stable_fpr
        + 2.0 * drift_fpr
        + 2.0 * worst_client_stable_fpr
        + 1.0 * worst_client_drift_fpr
        + 1000.0 * float(maximum_group_shortfall > 0.0)
        + 50.0 * mean_group_shortfall
        + 100.0 * maximum_group_shortfall
        + 0.05 * (1.0 - detection_rate)
    )
    return {
        "objective": objective,
        "stable_fpr": stable_fpr,
        "controlled_drift_fpr": drift_fpr,
        "worst_client_stable_fpr": worst_client_stable_fpr,
        "worst_client_controlled_drift_fpr": worst_client_drift_fpr,
        "detection_rate": detection_rate,
        "worst_group_detection": worst_group_detection,
        "mean_group_shortfall": mean_group_shortfall,
        "maximum_group_shortfall": maximum_group_shortfall,
    }


In [5]:
# Fixed swarm settings make the optimization repeatable.
PSO_SEED = 42
PSO_PARTICLES = 8
PSO_ITERATIONS = 10
PSO_INERTIA = 0.70
PSO_COGNITIVE = 1.50
PSO_SOCIAL = 1.50

# Normalized particle coordinates are decoded into these seven framework controls.
PARAMETER_NAMES = [
    "warmup_alpha",
    "stable_alpha",
    "drift_alpha",
    "anchor_strength",
    "confirmation_windows",
    "stable_batch_size",
    "threshold_ceiling",
]


def interpolate(value, lower, upper):
    """Map a normalized particle coordinate from [0, 1] to a linear range."""
    return lower + value * (upper - lower)


def log_interpolate(value, lower, upper):
    """Map a normalized coordinate to a log-scaled positive parameter range."""
    return 10 ** interpolate(value, np.log10(lower), np.log10(upper))


def decode_particle(position):
    """Convert one normalized PSO particle into framework hyperparameters."""
    return {
        "warmup_alpha": log_interpolate(position[0], 0.0005, 0.0500),
        "stable_alpha": log_interpolate(position[1], 0.00001, 0.00200),
        "drift_alpha": log_interpolate(position[2], 0.005, 0.100),
        "anchor_strength": interpolate(position[3], 0.15, 0.60),
        "confirmation_windows": int(round(interpolate(position[4], 8, 64))),
        "stable_batch_size": int(round(interpolate(position[5], 16, 128))),
        "threshold_ceiling": interpolate(position[6], 1.00, 1.60),
    }


def conservative_params():
    """Return a hand-set safe candidate that PSO must outperform to be selected."""
    return {
        "warmup_alpha": 0.002,
        "stable_alpha": 0.00001,
        "drift_alpha": 0.030,
        "anchor_strength": 0.35,
        "confirmation_windows": 32,
        "stable_batch_size": 64,
        "threshold_ceiling": 1.30,
    }


# Standard PSO loop: evaluate candidates, update bests, then move the swarm.
def run_pso(device_type, seed):
    """Tune one device type with standard particle-swarm position/velocity updates."""
    rng = np.random.default_rng(seed)
    dimensions = len(PARAMETER_NAMES)
    positions = rng.uniform(0.0, 1.0, size=(PSO_PARTICLES, dimensions))
    velocities = rng.uniform(-0.10, 0.10, size=(PSO_PARTICLES, dimensions))
    personal_best_positions = positions.copy()
    personal_best_scores = np.full(PSO_PARTICLES, np.inf)
    global_best_position = None
    global_best_score = np.inf
    global_best_metrics = None
    history = []

    baseline_params = conservative_params()
    baseline_metrics = simulate_candidate(device_type, baseline_params)
    history.append(
        {
            "device_type": device_type,
            "iteration": 0,
            "particle": 0,
            "candidate": "conservative",
            **baseline_params,
            **baseline_metrics,
        }
    )

    for iteration in range(1, PSO_ITERATIONS + 1):
        for particle_index in range(PSO_PARTICLES):
            params = decode_particle(positions[particle_index])
            metrics = simulate_candidate(device_type, params)
            history.append(
                {
                    "device_type": device_type,
                    "iteration": iteration,
                    "particle": particle_index + 1,
                    "candidate": "pso",
                    **params,
                    **metrics,
                }
            )

            if metrics["objective"] < personal_best_scores[particle_index]:
                personal_best_scores[particle_index] = metrics["objective"]
                personal_best_positions[particle_index] = positions[
                    particle_index
                ].copy()

            if metrics["objective"] < global_best_score:
                global_best_score = metrics["objective"]
                global_best_position = positions[particle_index].copy()
                global_best_metrics = metrics.copy()

        print(
            f"{device_type:>20} | iteration {iteration:>2}/{PSO_ITERATIONS} | "
            f"objective={global_best_score:.6f} | "
            f"stable FPR={global_best_metrics['stable_fpr']:.3%} | "
            f"drift FPR={global_best_metrics['controlled_drift_fpr']:.3%} | "
            f"worst attack={global_best_metrics['worst_group_detection']:.3%}"
        )

        r1 = rng.random(size=(PSO_PARTICLES, dimensions))
        r2 = rng.random(size=(PSO_PARTICLES, dimensions))
        velocities = (
            PSO_INERTIA * velocities
            + PSO_COGNITIVE * r1 * (personal_best_positions - positions)
            + PSO_SOCIAL * r2 * (global_best_position - positions)
        )
        positions = np.clip(positions + velocities, 0.0, 1.0)

    if baseline_metrics["objective"] <= global_best_score:
        return baseline_params, baseline_metrics, "conservative", pd.DataFrame(history)

    return (
        decode_particle(global_best_position),
        global_best_metrics,
        "pso",
        pd.DataFrame(history),
    )


## 5. Enroll and tune anchored client profiles

Using the selected PSO parameters, this section creates one adaptive profile per physical device/client. Each profile starts from the general device profile, processes trusted enrollment traffic, and then adapts on the disjoint benign tuning stream.

The calibrated profiles and compact states produced here are the starting point for the untouched final benign and attack evaluations.


In [6]:
# Optimize once per device type before constructing the final calibrated client profiles.
BEST_PARAMS_BY_TYPE = {}
pso_rows = []
pso_histories = []

for type_index, device_type in enumerate(general_profiles):
    params, metrics, selected_from, history = run_pso(
        device_type,
        PSO_SEED + type_index,
    )
    BEST_PARAMS_BY_TYPE[device_type] = params
    pso_rows.append(
        {
            "device_type": device_type,
            "selected_from": selected_from,
            **params,
            **metrics,
        }
    )
    pso_histories.append(history)

pso_summary = pd.DataFrame(pso_rows)
pso_history = pd.concat(pso_histories, ignore_index=True)
display(pso_summary)


def calibrate_clients():
    """Enroll each client, then adapt it on the disjoint benign tuning stream."""
    client_profiles = {}
    client_states = {}

    for client_id, enrollment_values in ENROLLMENT_BY_CLIENT.items():
        device_type = TYPE_BY_CLIENT[client_id]
        params = BEST_PARAMS_BY_TYPE[device_type]
        profile = new_client_profile(client_id, device_type)
        state = new_state(int(params["confirmation_windows"]))

        for values in enrollment_values:
            process_observation(
                profile,
                values,
                params,
                state,
                trusted=True,
            )
        flush_enrollment(profile, params, state)

        for values in TUNING_BY_CLIENT[client_id]:
            process_observation(
                profile,
                values,
                params,
                state,
            )

        client_profiles[client_id] = profile
        client_states[client_id] = state

    return client_profiles, client_states


calibrated_profiles, calibrated_states = calibrate_clients()


            doorbell | iteration  1/10 | objective=0.011618 | stable FPR=0.113% | drift FPR=0.147% | worst attack=100.000%
            doorbell | iteration  2/10 | objective=0.011191 | stable FPR=0.113% | drift FPR=0.135% | worst attack=100.000%
            doorbell | iteration  3/10 | objective=0.011191 | stable FPR=0.113% | drift FPR=0.135% | worst attack=100.000%
            doorbell | iteration  4/10 | objective=0.011191 | stable FPR=0.113% | drift FPR=0.135% | worst attack=100.000%
            doorbell | iteration  5/10 | objective=0.011191 | stable FPR=0.113% | drift FPR=0.135% | worst attack=100.000%
            doorbell | iteration  6/10 | objective=0.011191 | stable FPR=0.113% | drift FPR=0.135% | worst attack=100.000%
            doorbell | iteration  7/10 | objective=0.011191 | stable FPR=0.113% | drift FPR=0.135% | worst attack=100.000%
            doorbell | iteration  8/10 | objective=0.011191 | stable FPR=0.113% | drift FPR=0.135% | worst attack=100.000%
            door

,device_type,selected_from,warmup_alpha,stable_alpha,drift_alpha,anchor_strength,confirmation_windows,stable_batch_size,threshold_ceiling,objective,stable_fpr,controlled_drift_fpr,worst_client_stable_fpr,worst_client_controlled_drift_fpr,detection_rate,worst_group_detection,mean_group_shortfall,maximum_group_shortfall
0,doorbell,pso,0.000748,0.000275,0.051101,0.582978,24,58,1.252528,0.011191,0.001128,0.001354,0.001279,0.001413,1.0000,1.000,0.0,0.0
1,thermostat,pso,0.038835,0.000232,0.084269,0.599047,10,29,1.456719,0.009203,0.001526,0.000000,0.001526,0.000000,0.9990,0.998,0.0,0.0
2,baby_monitor,conservative,0.002000,0.000010,0.030000,0.350000,32,64,1.300000,0.000030,0.000000,0.000000,0.000000,0.000000,0.9994,0.998,0.0,0.0
3,security_camera,pso,0.030705,0.000043,0.036424,0.199451,40,48,1.264519,0.003618,0.000088,0.000088,0.001025,0.001025,0.9997,0.998,0.0,0.0
4,webcam,pso,0.000500,0.001123,0.076468,0.600000,60,127,1.349322,0.248514,0.040460,0.001918,0.040460,0.001918,1.0000,1.000,0.0,0.0


## 6. Run static Fisher, drift, adaptive Fisher, and attack tests

The final experiment compares three benign conditions:

- **Baseline:** fixed general profiles on the early baseline partition.
- **Drift Simulation:** the same fixed profiles on the later client stream, showing false positives without adaptation.
- **Adaptive Fisher-Weighted Z:** client-specific adaptive profiles on that same untouched final benign stream.

The untouched attack-test groups are then scored using both static and final adaptive profiles. **Attack windows never call the online update path**, so malicious traffic cannot alter the client statistics during attack evaluation.

Both benign and attack outputs use the same causal 2-of-3 evidence decision.


In [7]:
def run_static_benign(dataframe, model_name):
    """Score benign windows with fixed general profiles and apply 2-of-3 evidence."""
    rows_out = []
    ordered = dataframe.sort_values(
        ["device_name", "position"],
        kind="stable",
    )

    for _, row in ordered.iterrows():
        start = time.perf_counter()
        profile = general_profiles[row["device_type"]]
        values = row[FEATURE_COLUMNS].to_numpy(dtype=float)
        score = score_general(profile, values)
        rows_out.append(
            {
                "model": model_name,
                "device_name": row["device_name"],
                "device_type": row["device_type"],
                "position": int(row["position"]),
                "score": score,
                "threshold": profile.threshold,
                "flagged": profile.is_anomalous(score),
                "total_time_ms": (time.perf_counter() - start) * 1000,
            }
        )

    return add_evidence_flags(
        pd.DataFrame(rows_out),
        group_columns=["device_name"],
        order_column="position",
    )


def run_adaptive_benign():
    """Process the untouched benign stream with online client adaptation."""
    rows_out = []
    ordered = stream_df.sort_values(
        ["device_name", "position"],
        kind="stable",
    )

    for _, row in ordered.iterrows():
        start = time.perf_counter()
        client_id = row["device_name"]
        device_type = row["device_type"]
        profile = calibrated_profiles[client_id]
        state = calibrated_states[client_id]
        params = BEST_PARAMS_BY_TYPE[device_type]
        values = row[FEATURE_COLUMNS].to_numpy(dtype=float)
        adaptive = process_observation(
            profile,
            values,
            params,
            state,
        )

        rows_out.append(
            {
                "model": "Adaptive Fisher-Weighted Z",
                "device_name": client_id,
                "device_type": device_type,
                "position": int(row["position"]),
                "score": adaptive["score"],
                "threshold": adaptive["threshold_before"],
                "flagged": bool(adaptive["flagged"]),
                "updated": adaptive["updated"],
                "update_action": adaptive["update_action"],
                "drift_confirmed": adaptive["drift_confirmed"],
                "total_time_ms": (time.perf_counter() - start) * 1000,
            }
        )

    return add_evidence_flags(
        pd.DataFrame(rows_out),
        group_columns=["device_name"],
        order_column="position",
    )


# Run the three benign conditions first, then evaluate the reserved attack-test groups.
baseline_results = run_static_benign(
    baseline_test_df,
    "Baseline",
)
drift_results = run_static_benign(
    stream_df,
    "Drift Simulation",
)
adaptive_results = run_adaptive_benign()


def evidence_flags_array(flags):
    """Apply the same causal two-of-three decision to one attack CSV."""
    flags = np.asarray(flags, dtype=bool)
    hits = np.convolve(
        flags.astype(int),
        np.ones(EVIDENCE_WINDOW, dtype=int),
        mode="full",
    )[: len(flags)]
    return hits >= EVIDENCE_MIN_HITS


def score_general_matrix(profile, values):
    """Vectorized scoring of attack windows with a fixed general profile."""
    means, stds = profile_arrays(profile)
    return fisher_weighted_scores(
        values,
        means,
        stds,
        profile_weights(profile),
    )


# Important: this function scores attacks against frozen final profiles; it never adapts on them.
def run_attack_test(model_name):
    """Score the untouched attack-test groups without updating client profiles."""
    summaries = []

    for group in attack_test_groups:
        client_id = group["device_name"]
        device_type = group["device_type"]
        values = np.asarray(group["values"], dtype=float)

        if model_name == "Drift Simulation":
            profile = general_profiles[device_type]
            scores = score_general_matrix(profile, values)
        else:
            profile = calibrated_profiles[client_id]
            params = BEST_PARAMS_BY_TYPE[device_type]
            scores = score_client_matrix(profile, values, params)

        raw_flags = scores > profile.threshold
        evidence_flags = evidence_flags_array(raw_flags)
        summaries.append(
            {
                "model": model_name,
                "device_name": client_id,
                "device_type": device_type,
                "source_file": group["source_file"],
                "attack_family": group["attack_family"],
                "attack_type": group["attack_type"],
                "raw_detected_windows": int(raw_flags.sum()),
                "detected_windows": int(evidence_flags.sum()),
                "total_windows": len(raw_flags),
                "raw_detection_rate": float(raw_flags.mean()),
                "detection_rate": float(evidence_flags.mean()),
            }
        )

    return pd.DataFrame(summaries)


static_attack_summary = run_attack_test("Drift Simulation")
adaptive_attack_summary = run_attack_test("Adaptive Fisher-Weighted Z")


## 7. Report and save compact final results

This section summarizes final evidence-level benign FPR, attack-window detection, per-device/per-type breakdowns, runtime, compact numeric storage overhead, and the selected PSO parameters.

The result CSVs are written to the configured N-BaIoT results directory so the reported experiment can be inspected without rerunning optimization.


In [8]:
def evidence_fpr(results):
    """Return the final benign false-positive rate after the 2-of-3 evidence rule."""
    return float(results["evidence_flagged"].mean())


def overall_attack_detection(summary):
    """Aggregate detected attack windows into one overall detection rate."""
    return float(summary["detected_windows"].sum() / summary["total_windows"].sum())


# Aggregate headline FPR/detection metrics before detailed device and attack breakdowns.
overall_results = pd.DataFrame(
    [
        {
            "dataset": "N-BaIoT",
            "experiment": "Baseline",
            "benign_windows": len(baseline_results),
            "fpr": evidence_fpr(baseline_results),
            "attack_windows": np.nan,
            "detection_rate": np.nan,
        },
        {
            "dataset": "N-BaIoT",
            "experiment": "Drift Simulation",
            "benign_windows": len(drift_results),
            "fpr": evidence_fpr(drift_results),
            "attack_windows": static_attack_summary["total_windows"].sum(),
            "detection_rate": overall_attack_detection(static_attack_summary),
        },
        {
            "dataset": "N-BaIoT",
            "experiment": "Adaptive Fisher-Weighted Z",
            "benign_windows": len(adaptive_results),
            "fpr": evidence_fpr(adaptive_results),
            "attack_windows": adaptive_attack_summary["total_windows"].sum(),
            "detection_rate": overall_attack_detection(adaptive_attack_summary),
        },
    ]
)

all_benign_results = pd.concat(
    [baseline_results, drift_results, adaptive_results],
    ignore_index=True,
)
device_type_fpr = (
    all_benign_results.groupby(
        ["model", "device_type"],
        sort=False,
    )["evidence_flagged"]
    .agg(
        false_flags="sum",
        total_windows="count",
        fpr="mean",
    )
    .reset_index()
)

device_fpr = (
    all_benign_results.groupby(
        ["model", "device_name", "device_type"],
        sort=False,
    )["evidence_flagged"]
    .agg(
        false_flags="sum",
        total_windows="count",
        fpr="mean",
    )
    .reset_index()
)

attack_group_detection = pd.concat(
    [static_attack_summary, adaptive_attack_summary],
    ignore_index=True,
)
attack_type_detection = (
    attack_group_detection.groupby(
        ["model", "attack_family", "attack_type"],
        sort=False,
    )
    .agg(
        detected_windows=("detected_windows", "sum"),
        total_windows=("total_windows", "sum"),
    )
    .reset_index()
)
attack_type_detection["detection_rate"] = (
    attack_type_detection["detected_windows"]
    / attack_type_detection["total_windows"]
)

# Runtime statistics come from the final per-window benign evaluation paths.
runtime_summary = (
    all_benign_results.groupby(
        "model",
        sort=False,
    )["total_time_ms"]
    .agg(
        average_ms="mean",
        median_ms="median",
        maximum_ms="max",
    )
    .reset_index()
)

# Estimate raw numeric state only; Python object/dictionary overhead is excluded.
FLOAT64_BYTES = np.dtype(np.float64).itemsize
PROFILE_NUMERIC_VALUES = 3 * len(FEATURE_COLUMNS) + 1
PROFILE_NUMERIC_BYTES = PROFILE_NUMERIC_VALUES * FLOAT64_BYTES

# One shared running accumulator stores count, mean, and M2. During confirmed
# drift it additionally retains only one scalar score per candidate window.
RUNNING_MOMENT_BYTES = (
    1 + 2 * len(FEATURE_COLUMNS)
) * FLOAT64_BYTES
MAX_CONFIRMATION_WINDOWS = max(
    int(params["confirmation_windows"])
    for params in BEST_PARAMS_BY_TYPE.values()
)
PEAK_DRIFT_STATE_BYTES = (
    RUNNING_MOMENT_BYTES
    + MAX_CONFIRMATION_WINDOWS * FLOAT64_BYTES
)

storage_summary = pd.DataFrame(
    [
        {
            "storage_scope": "general profile statistical state",
            "items": len(general_profiles),
            "bytes_per_item": PROFILE_NUMERIC_BYTES,
            "total_bytes": len(general_profiles) * PROFILE_NUMERIC_BYTES,
            "details": (
                f"{len(FEATURE_COLUMNS)} means + "
                f"{len(FEATURE_COLUMNS)} standard deviations + "
                f"{len(FEATURE_COLUMNS)} Fisher weights + 1 threshold"
            ),
        },
        {
            "storage_scope": "client profile statistical state",
            "items": len(calibrated_profiles),
            "bytes_per_item": PROFILE_NUMERIC_BYTES,
            "total_bytes": len(calibrated_profiles) * PROFILE_NUMERIC_BYTES,
            "details": (
                f"{len(FEATURE_COLUMNS)} means + "
                f"{len(FEATURE_COLUMNS)} standard deviations + "
                f"{len(FEATURE_COLUMNS)} Fisher weights + 1 threshold"
            ),
        },
        {
            "storage_scope": "stable rolling accumulator",
            "items": len(calibrated_profiles),
            "bytes_per_item": RUNNING_MOMENT_BYTES,
            "total_bytes": len(calibrated_profiles) * RUNNING_MOMENT_BYTES,
            "details": (
                f"count + {len(FEATURE_COLUMNS)} running means + "
                f"{len(FEATURE_COLUMNS)} running M2 values; "
                "stable batch closes at a PSO-selected size of "
                f"{min(int(p['stable_batch_size']) for p in BEST_PARAMS_BY_TYPE.values())}-"
                f"{max(int(p['stable_batch_size']) for p in BEST_PARAMS_BY_TYPE.values())} windows across device types"
            ),
        },
        {
            "storage_scope": "peak drift rolling state",
            "items": len(calibrated_profiles),
            "bytes_per_item": PEAK_DRIFT_STATE_BYTES,
            "total_bytes": len(calibrated_profiles) * PEAK_DRIFT_STATE_BYTES,
            "details": (
                f"running moments + up to {MAX_CONFIRMATION_WINDOWS} "
                "scalar drift scores; no full feature windows retained"
            ),
        },
    ]
)

outputs = {
    "overall_results.csv": overall_results,
    "scoring_summary.csv": general_profile_summary,
    "fisher_feature_weights.csv": fisher_feature_weights,
    "device_type_fpr.csv": device_type_fpr,
    "device_fpr.csv": device_fpr,
    "attack_type_detection.csv": attack_type_detection,
    "attack_group_detection.csv": attack_group_detection,
    "pso_summary.csv": pso_summary,
    "pso_history.csv": pso_history,
    "runtime_summary.csv": runtime_summary,
    "storage_summary.csv": storage_summary,
    "split_summary.csv": split_summary,
}
for filename, dataframe in outputs.items():
    dataframe.to_csv(RESULTS_DIR / filename, index=False)

print("Adaptive Fisher-Weighted Z results use the deployed 2-of-3 evidence decision.")

display(overall_results)
display(device_type_fpr)
display(attack_type_detection)
display(runtime_summary)
display(storage_summary)


Adaptive Fisher-Weighted Z results use the deployed 2-of-3 evidence decision.


,dataset,experiment,benign_windows,fpr,attack_windows,detection_rate
0,N-BaIoT,Baseline,55594,0.027107,NaN,NaN
1,N-BaIoT,Drift Simulation,222376,0.015244,5205371.0,0.99990
2,N-BaIoT,Adaptive Fisher-Weighted Z,222376,0.011561,5205371.0,0.99971


,model,device_type,false_flags,total_windows,fpr
0,Baseline,doorbell,189,8865,0.021320
1,Baseline,thermostat,0,1312,0.000000
2,Baseline,baby_monitor,1,17524,0.000057
3,Baseline,security_camera,56,22678,0.002469
4,Baseline,webcam,1261,5215,0.241802
5,Drift Simulation,doorbell,511,35460,0.014411
6,Drift Simulation,thermostat,9,5246,0.001716
7,Drift Simulation,baby_monitor,4,70096,0.000057
8,Drift Simulation,security_camera,1196,90714,0.013184
9,Drift Simulation,webcam,1670,20860,0.080058


,model,attack_family,attack_type,detected_windows,total_windows,detection_rate
0,Drift Simulation,Gafgyt,combo,412075,412129,0.999869
1,Drift Simulation,Gafgyt,junk,209266,209436,0.999188
2,Drift Simulation,Gafgyt,scan,204065,204093,0.999863
3,Drift Simulation,Gafgyt,tcp,687813,687882,0.999900
4,Drift Simulation,Gafgyt,udp,757036,757096,0.999921
5,Drift Simulation,Mirai,ack,515026,515059,0.999936
6,Drift Simulation,Mirai,scan,430375,430385,0.999977
7,Drift Simulation,Mirai,syn,586601,586642,0.999930
8,Drift Simulation,Mirai,udp,983980,984003,0.999977
9,Drift Simulation,Mirai,udpplain,418614,418646,0.999924


,model,average_ms,median_ms,maximum_ms
0,Baseline,0.203682,0.1994,0.6427
1,Drift Simulation,0.203102,0.1979,9.0104
2,Adaptive Fisher-Weighted Z,0.289188,0.2811,1.5380


,storage_scope,items,bytes_per_item,total_bytes,details
0,general profile statistical state,5,560,2800,23 means + 23 standard deviations + 23 Fisher ...
1,client profile statistical state,9,560,5040,23 means + 23 standard deviations + 23 Fisher ...
2,stable rolling accumulator,9,376,3384,count + 23 running means + 23 running M2 value...
3,peak drift rolling state,9,856,7704,running moments + up to 60 scalar drift scores...
